### 第一步：判断是否连续

In [11]:
import re

def _getIdx(node_name):
	### eg. '/model.0/conv/Conv' -> '0'\
	m = re.search(r'model\.(\d+)', node_name)
	idx = int(m.group(1)) 
	return idx

In [12]:
from collections import Counter
from itertools import groupby

def _checkContinuous(arr):
	### arr: [(node_name, weight_size), ...]
	continuousArr = []

	# 第一步：统计每个idx出现次数。
	count = Counter(_getIdx(node_name) for node_name, _ in arr)

    #第二步：若idx出现次数大于1，且连续出现，则作为数组加入continousArr。
	for idx, group in groupby(arr, key=lambda x: _getIdx(x[0])):
		group = list(group)
		if count[idx] > 1 and len(group) == count[idx]:
			continuousArr.append(group)
		else:
			continuousArr.extend(group)

    # 第三步：返回continuousArr
	return continuousArr

### 简单测试
arr = [('/model.0/conv/Conv', 10), ('/model.2/conv/Conv', 20), ('/model.2/conv/Conv', 30), ('/model.1/conv/Conv', 40)]
print(_checkContinuous(arr))

[('/model.0/conv/Conv', 10), [('/model.2/conv/Conv', 20), ('/model.2/conv/Conv', 30)], ('/model.1/conv/Conv', 40)]


### 测试用例1

In [13]:
# def t(i, s, w=0):                      # 简写：t(0,'a') → ('/model.0/a', 0)
#     return (f'/model.{i}/{s}', w)

# cases = [
#     # 1. 不连续（核心分支）：0 被 1 隔开
#     ([t(0,'a'), t(1,'a'), t(0,'b')],
#      [t(0,'a'), t(1,'a'), t(0,'b')]),

#     # 2. 不连续段长 > 1：0 的前一段有 2 个，但总数是 3，这一段也要逐个拆开
#     ([t(0,'a'), t(0,'b'), t(1,'a'), t(0,'c')],
#      [t(0,'a'), t(0,'b'), t(1,'a'), t(0,'c')]),

#     # 3. 连续与不连续混合
#     ([t(0,'a'), t(0,'b'), t(1,'a'), t(2,'a'), t(1,'b')],
#      [[t(0,'a'), t(0,'b')], t(1,'a'), t(2,'a'), t(1,'b')]),

#     # 4. 多位数 idx：model.10 不能被当成 model.1
#     ([t(1,'a'), t(10,'a'), t(1,'b')],
#      [t(1,'a'), t(10,'a'), t(1,'b')]),

#     # 5. 同一 idx 下节点名不同（真实 ONNX 场景）
#     ([('/model.2/conv/Conv',1), ('/model.2/act/Sigmoid',2)],
#      [[('/model.2/conv/Conv',1), ('/model.2/act/Sigmoid',2)]]),

#     # 6. 全部相同
#     ([t(3,'a'), t(3,'b'), t(3,'c')],
#      [[t(3,'a'), t(3,'b'), t(3,'c')]]),

#     # 7. 空列表
#     ([], []),
# ]

# for arr, expected in cases:
#     got = checkContinuous(arr)
#     assert got == expected, f'\n got:      {got}\n expected: {expected}'
# print('all passed')

### 第二步：计算百分比
#### getClosestNodes(arr, target) -> closestNodes

arr: [(node_name, weight_size), ...]

In [14]:
def _reshape(obj):
    ### [(x1, y1), (x2, y2), ...] -> [ [x1, x2, ...], y1+y2+... ]
    if isinstance(obj, list):
        x, y = [], 0
        for i in obj:
            x.append(i[0])
            y += i[1]
    else:
        x, y = obj[0], obj[1]

    return [x, y]

In [15]:
def getClosestNodes(arr, target):
    arr = [tuple(p) for p in arr]       
    continuousArr = _checkContinuous(arr)
    reshapedArr = [_reshape(i) for i in continuousArr]

    closestNodes, closestParam = [], 0

    for k, (nodes, param) in enumerate(reshapedArr):
        if abs(target - closestParam - param) < abs(target - closestParam):
            closestParam += param
            closestNodes.append(nodes)
        else:
            break

    closestNodes = [y for x in closestNodes for y in (x if isinstance(x, list) else [x])]
    return closestNodes

### 简单测试
# arr = [('/model.0/conv/Conv', 10), [('/model.2/conv/Conv', 20), ('/model.2/conv/Conv', 30)], ('/model.1/conv/Conv', 40)]
# target = 70
# print(getClosestNodes(arr, target))

### 测试用例2

In [16]:
# A = ('a', 10)
# G = [('b', 20), ('c', 30)]          # 组，合计 50
# D = ('d', 40)
# base = [A, G, D]                    # 前缀和：10, 60, 100

# cases = [
#     # 1. 原用例：60 比 100 更接近 70
#     (base, 70,   ['a', 'b', 'c']),
#     # 2. 恰好相等：差为 0 后，后面的元素不可能更近
#     (base, 60,   ['a', 'b', 'c']),
#     # 3. 全部取完：循环正常结束，没有触发 break
#     (base, 1000, ['a', 'b', 'c', 'd']),
#     # 4. 一个都不取：|4-10|=6 > |4-0|=4
#     (base, 4,    []),
#     # 5. 平局：|80-60| == |80-100|，按当前代码保留较短的那个
#     (base, 80,   ['a', 'b', 'c']),
#     # 6. 空数组
#     ([],   50,   []),
#     # 7. 中间有 n=0：最优是全部取，和为 50
#     ([A, ('z', 0), D], 50, ['a', 'z', 'd']),
#     # 8. 组必须整体取：target=30 时，取半个组 [a,b] 正好是 30，但这是不允许的
#     (base, 30,   ['a']),
# ]

# for arr, target, expected in cases:
#     got = getClosestNodes(arr, target)
#     assert got == expected, (target, got, expected)
# print('all passed')

### 第三步：组装

In [17]:
def calTargetParam(targetRatio, totalParam):
	return targetRatio * totalParam
# (1 - targetRatio) / 0.75 * totalParam

In [18]:
from collections import defaultdict
import numpy as np
from itertools import product

def getQuantizedNodes(arrs, total_param):
	NODES = defaultdict(list)

	configs = []
	for (name, arr), targetRatio20 in product(arrs.items(), range(1, 20)):
		targetRatio = targetRatio20 / 20
		targetParam = calTargetParam(targetRatio, total_param)
		got = getClosestNodes(arr, targetParam)
		NODES[tuple(got)].append((targetRatio, name))

		configs.append(
			{
			'targetRatio': targetRatio,
			'targetParam': targetParam,
			'got': got,
			'name': name
			}
		)

	with open('configs.json', 'w') as f:
		import json
		json.dump(configs, f, indent=4)

	return NODES

In [19]:
arr_block_layer = [['/model.0/conv/Conv', 432], ['/model.7/conv/Conv', 294912], ['/model.5/conv/Conv', 147456], ['/model.20/conv/Conv', 147456], ['/model.9/cv2/conv/Conv', 131072], ['/model.9/cv1/conv/Conv', 32768], ['/model.17/conv/Conv', 36864], ['/model.3/conv/Conv', 36864], ['/model.22/cv1/conv/Conv', 98304], ['/model.22/cv2/conv/Conv', 98304], ['/model.22/m.0/m.0.0/cv2/conv/Conv', 73728], ['/model.22/m.0/m.0.0/cv1/conv/Conv', 73728], ['/model.22/m.0/m.0.1/attn/qkv/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.0/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.1/conv/Conv', 32768], ['/model.8/cv2/conv/Conv', 98304], ['/model.8/cv1/conv/Conv', 65536], ['/model.8/m.0/m/m.0/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv2/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.0/cv2/conv/Conv', 36864], ['/model.22/m.0/m.0.1/attn/proj/conv/Conv', 16384], ['/model.8/m.0/cv3/conv/Conv', 16384], ['/model.8/m.0/cv2/conv/Conv', 8192], ['/model.8/m.0/cv1/conv/Conv', 8192], ['/model.10/cv1/conv/Conv', 65536], ['/model.10/cv2/conv/Conv', 65536], ['/model.10/m/m.0/ffn/ffn.1/conv/Conv', 32768], ['/model.10/m/m.0/attn/qkv/conv/Conv', 32768], ['/model.10/m/m.0/ffn/ffn.0/conv/Conv', 32768], ['/model.10/m/m.0/attn/proj/conv/Conv', 16384], ['/model.13/cv1/conv/Conv', 49152], ['/model.13/cv2/conv/Conv', 24576], ['/model.13/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.19/cv1/conv/Conv', 24576], ['/model.19/cv2/conv/Conv', 24576], ['/model.23/one2one_cv4.2/one2one_cv4.2.0/conv/Conv', 73728], ['/model.23/proto/cv2/conv/Conv', 36864], ['/model.23/proto/cv1/conv/Conv', 36864], ['/model.23/proto/feat_fuse/conv/Conv', 36864], ['/model.23/one2one_cv4.1/one2one_cv4.1.0/conv/Conv', 36864], ['/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv', 20480], ['/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv', 18432], ['/model.23/one2one_cv4.0/one2one_cv4.0.0/conv/Conv', 18432], ['/model.23/proto/feat_refine.1/conv/Conv', 16384], ['/model.19/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv', 10240], ['/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv', 9216], ['/model.23/one2one_cv4.1/one2one_cv4.1.1/conv/Conv', 9216], ['/model.23/one2one_cv4.0/one2one_cv4.0.1/conv/Conv', 9216], ['/model.23/one2one_cv4.2/one2one_cv4.2.1/conv/Conv', 9216], ['/model.6/cv2/conv/Conv', 24576], ['/model.13/m.0/cv3/conv/Conv', 4096], ['/model.23/proto/feat_refine.0/conv/Conv', 8192], ['/model.6/cv1/conv/Conv', 16384], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv', 6400], ['/model.6/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv', 5120], ['/model.22/m.0/m.0.1/attn/pe/conv/Conv', 1152], ['/model.19/m.0/cv3/conv/Conv', 4096], ['/model.6/m.0/cv3/conv/Conv', 4096], ['/model.4/cv2/conv/Conv', 12288], ['/model.4/m.0/cv2/conv/Conv', 4608], ['/model.4/m.0/cv1/conv/Conv', 4608], ['/model.4/cv1/conv/Conv', 4096], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv', 2304], ['/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv', 2304], ['/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv', 2304], ['/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv', 2304], ['/model.13/m.0/cv2/conv/Conv', 2048], ['/model.13/m.0/cv1/conv/Conv', 2048], ['/model.19/m.0/cv2/conv/Conv', 2048], ['/model.19/m.0/cv1/conv/Conv', 2048], ['/model.23/proto/cv3/conv/Conv', 2048], ['/model.6/m.0/cv2/conv/Conv', 2048], ['/model.6/m.0/cv1/conv/Conv', 2048], ['/model.1/conv/Conv', 4608], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv', 1152], ['/model.10/m/m.0/attn/pe/conv/Conv', 1152], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv', 720], ['/model.16/cv1/conv/Conv', 16384], ['/model.16/cv2/conv/Conv', 6144], ['/model.16/m.0/m/m.0/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv2/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.0/cv2/conv/Conv', 2304], ['/model.16/m.0/cv3/conv/Conv', 1024], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv', 576], ['/model.2/cv2/conv/Conv', 3072], ['/model.2/m.0/cv2/conv/Conv', 1152], ['/model.2/m.0/cv1/conv/Conv', 1152], ['/model.2/cv1/conv/Conv', 1024], ['/model.16/m.0/cv2/conv/Conv', 512], ['/model.16/m.0/cv1/conv/Conv', 512]]
arr_block = [['/model.0/conv/Conv', 432], ['/model.7/conv/Conv', 294912], ['/model.5/conv/Conv', 147456], ['/model.20/conv/Conv', 147456], ['/model.9/cv2/conv/Conv', 131072], ['/model.9/cv1/conv/Conv', 32768], ['/model.17/conv/Conv', 36864], ['/model.3/conv/Conv', 36864], ['/model.22/cv2/conv/Conv', 98304], ['/model.22/cv1/conv/Conv', 98304], ['/model.22/m.0/m.0.1/ffn/ffn.1/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.0/conv/Conv', 32768], ['/model.22/m.0/m.0.1/attn/proj/conv/Conv', 16384], ['/model.22/m.0/m.0.1/attn/pe/conv/Conv', 1152], ['/model.22/m.0/m.0.1/attn/qkv/conv/Conv', 32768], ['/model.22/m.0/m.0.0/cv2/conv/Conv', 73728], ['/model.22/m.0/m.0.0/cv1/conv/Conv', 73728], ['/model.8/m.0/cv2/conv/Conv', 8192], ['/model.8/m.0/m/m.1/cv1/conv/Conv', 36864], ['/model.8/m.0/cv3/conv/Conv', 16384], ['/model.8/m.0/m/m.1/cv2/conv/Conv', 36864], ['/model.8/m.0/cv1/conv/Conv', 8192], ['/model.8/m.0/m/m.0/cv2/conv/Conv', 36864], ['/model.8/cv2/conv/Conv', 98304], ['/model.8/cv1/conv/Conv', 65536], ['/model.8/m.0/m/m.0/cv1/conv/Conv', 36864], ['/model.10/m/m.0/attn/pe/conv/Conv', 1152], ['/model.10/m/m.0/ffn/ffn.1/conv/Conv', 32768], ['/model.10/cv1/conv/Conv', 65536], ['/model.10/m/m.0/attn/qkv/conv/Conv', 32768], ['/model.10/m/m.0/ffn/ffn.0/conv/Conv', 32768], ['/model.10/m/m.0/attn/proj/conv/Conv', 16384], ['/model.10/cv2/conv/Conv', 65536], ['/model.13/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.13/m.0/cv1/conv/Conv', 2048], ['/model.13/m.0/cv2/conv/Conv', 2048], ['/model.13/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.13/cv1/conv/Conv', 49152], ['/model.13/cv2/conv/Conv', 24576], ['/model.13/m.0/cv3/conv/Conv', 4096], ['/model.19/m.0/cv1/conv/Conv', 2048], ['/model.19/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.19/cv1/conv/Conv', 24576], ['/model.19/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/cv3/conv/Conv', 4096], ['/model.19/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.19/cv2/conv/Conv', 24576], ['/model.19/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.19/m.0/cv2/conv/Conv', 2048], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv', 576], ['/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv', 9216], ['/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv', 2304], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv', 5120], ['/model.23/one2one_cv4.0/one2one_cv4.0.0/conv/Conv', 18432], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv', 6400], ['/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv', 18432], ['/model.23/one2one_cv4.0/one2one_cv4.0.1/conv/Conv', 9216], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv', 1152], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv', 720], ['/model.23/one2one_cv4.1/one2one_cv4.1.0/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv', 2304], ['/model.23/one2one_cv4.2/one2one_cv4.2.0/conv/Conv', 73728], ['/model.23/proto/feat_refine.1/conv/Conv', 16384], ['/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv', 2304], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv', 20480], ['/model.23/one2one_cv4.2/one2one_cv4.2.1/conv/Conv', 9216], ['/model.23/proto/feat_fuse/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv', 720], ['/model.23/proto/cv1/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv', 6400], ['/model.23/proto/cv2/conv/Conv', 36864], ['/model.23/proto/cv3/conv/Conv', 2048], ['/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv', 2304], ['/model.23/proto/feat_refine.0/conv/Conv', 8192], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv', 10240], ['/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv', 36864], ['/model.23/one2one_cv4.1/one2one_cv4.1.1/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.6/m.0/cv1/conv/Conv', 2048], ['/model.6/m.0/cv2/conv/Conv', 2048], ['/model.6/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.6/m.0/cv3/conv/Conv', 4096], ['/model.6/cv2/conv/Conv', 24576], ['/model.6/cv1/conv/Conv', 16384], ['/model.6/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.4/cv2/conv/Conv', 12288], ['/model.4/cv1/conv/Conv', 4096], ['/model.4/m.0/cv1/conv/Conv', 4608], ['/model.4/m.0/cv2/conv/Conv', 4608], ['/model.1/conv/Conv', 4608], ['/model.16/m.0/m/m.0/cv1/conv/Conv', 2304], ['/model.16/cv1/conv/Conv', 16384], ['/model.16/m.0/m/m.0/cv2/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv2/conv/Conv', 2304], ['/model.16/cv2/conv/Conv', 6144], ['/model.16/m.0/cv3/conv/Conv', 1024], ['/model.16/m.0/cv1/conv/Conv', 512], ['/model.16/m.0/cv2/conv/Conv', 512], ['/model.2/m.0/cv1/conv/Conv', 1152], ['/model.2/cv1/conv/Conv', 1024], ['/model.2/cv2/conv/Conv', 3072], ['/model.2/m.0/cv2/conv/Conv', 1152]]
arr_layer = [['/model.0/conv/Conv', 432], ['/model.7/conv/Conv', 294912], ['/model.5/conv/Conv', 147456], ['/model.20/conv/Conv', 147456], ['/model.9/cv2/conv/Conv', 131072], ['/model.22/cv1/conv/Conv', 98304], ['/model.8/cv2/conv/Conv', 98304], ['/model.22/cv2/conv/Conv', 98304], ['/model.22/m.0/m.0.0/cv2/conv/Conv', 73728], ['/model.22/m.0/m.0.0/cv1/conv/Conv', 73728], ['/model.23/one2one_cv4.2/one2one_cv4.2.0/conv/Conv', 73728], ['/model.10/cv1/conv/Conv', 65536], ['/model.8/cv1/conv/Conv', 65536], ['/model.10/cv2/conv/Conv', 65536], ['/model.13/cv1/conv/Conv', 49152], ['/model.8/m.0/m/m.0/cv1/conv/Conv', 36864], ['/model.23/proto/cv2/conv/Conv', 36864], ['/model.23/proto/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv2/conv/Conv', 36864], ['/model.17/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.0/cv2/conv/Conv', 36864], ['/model.23/proto/feat_fuse/conv/Conv', 36864], ['/model.23/one2one_cv4.1/one2one_cv4.1.0/conv/Conv', 36864], ['/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv', 36864], ['/model.3/conv/Conv', 36864], ['/model.10/m/m.0/ffn/ffn.1/conv/Conv', 32768], ['/model.22/m.0/m.0.1/attn/qkv/conv/Conv', 32768], ['/model.10/m/m.0/attn/qkv/conv/Conv', 32768], ['/model.9/cv1/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.0/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.1/conv/Conv', 32768], ['/model.10/m/m.0/ffn/ffn.0/conv/Conv', 32768], ['/model.19/cv1/conv/Conv', 24576], ['/model.13/cv2/conv/Conv', 24576], ['/model.6/cv2/conv/Conv', 24576], ['/model.19/cv2/conv/Conv', 24576], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv', 20480], ['/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv', 18432], ['/model.23/one2one_cv4.0/one2one_cv4.0.0/conv/Conv', 18432], ['/model.6/cv1/conv/Conv', 16384], ['/model.10/m/m.0/attn/proj/conv/Conv', 16384], ['/model.16/cv1/conv/Conv', 16384], ['/model.8/m.0/cv3/conv/Conv', 16384], ['/model.22/m.0/m.0.1/attn/proj/conv/Conv', 16384], ['/model.23/proto/feat_refine.1/conv/Conv', 16384], ['/model.4/cv2/conv/Conv', 12288], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv', 10240], ['/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv', 9216], ['/model.19/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.19/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.23/one2one_cv4.1/one2one_cv4.1.1/conv/Conv', 9216], ['/model.23/one2one_cv4.0/one2one_cv4.0.1/conv/Conv', 9216], ['/model.23/one2one_cv4.2/one2one_cv4.2.1/conv/Conv', 9216], ['/model.8/m.0/cv2/conv/Conv', 8192], ['/model.8/m.0/cv1/conv/Conv', 8192], ['/model.23/proto/feat_refine.0/conv/Conv', 8192], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv', 6400], ['/model.16/cv2/conv/Conv', 6144], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv', 5120], ['/model.4/m.0/cv2/conv/Conv', 4608], ['/model.1/conv/Conv', 4608], ['/model.4/m.0/cv1/conv/Conv', 4608], ['/model.19/m.0/cv3/conv/Conv', 4096], ['/model.6/m.0/cv3/conv/Conv', 4096], ['/model.4/cv1/conv/Conv', 4096], ['/model.13/m.0/cv3/conv/Conv', 4096], ['/model.2/cv2/conv/Conv', 3072], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv', 2304], ['/model.16/m.0/m/m.0/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv2/conv/Conv', 2304], ['/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv', 2304], ['/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.0/cv2/conv/Conv', 2304], ['/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv', 2304], ['/model.22/m.0/m.0.1/attn/pe/conv/Conv', 1152], ['/model.23/proto/cv3/conv/Conv', 2048], ['/model.13/m.0/cv2/conv/Conv', 2048], ['/model.6/m.0/cv2/conv/Conv', 2048], ['/model.19/m.0/cv2/conv/Conv', 2048], ['/model.13/m.0/cv1/conv/Conv', 2048], ['/model.6/m.0/cv1/conv/Conv', 2048], ['/model.19/m.0/cv1/conv/Conv', 2048], ['/model.2/m.0/cv2/conv/Conv', 1152], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv', 1152], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv', 720], ['/model.10/m/m.0/attn/pe/conv/Conv', 1152], ['/model.2/m.0/cv1/conv/Conv', 1152], ['/model.2/cv1/conv/Conv', 1024], ['/model.16/m.0/cv3/conv/Conv', 1024], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv', 576], ['/model.16/m.0/cv2/conv/Conv', 512], ['/model.16/m.0/cv1/conv/Conv', 512]]
total_param = 2765036
arrs = {'arr_block': arr_block, 'arr_layer': arr_layer, 'arr_block_layer': arr_block_layer}

NODES = getQuantizedNodes(arrs, total_param)